In [20]:
# Minimal setup
from pathlib import Path
import re
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "working_data" / "nhamcs_data_2018_22.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path exists: {DATA_PATH.exists()}")

Project root: /home/gaurav/python/kaggle/triage
Data path exists: True


In [21]:
# Load CSV and keep only complaint text + acuity target
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()

TEXT_COL = "chief_complaint_text"
TARGET_CANDIDATES = ["target_triage_acuity", "triage_acuity", "target"]
target_col = next((c for c in TARGET_CANDIDATES if c in df.columns), None)

if target_col is None:
    raise KeyError(
        "Target column not found. "
        f"Tried {TARGET_CANDIDATES}. First columns: {list(df.columns[:15])}"
    )

data = (
    df[[TEXT_COL, target_col]]
    .rename(columns={target_col: "target_triage_acuity"})
    .dropna(subset=[TEXT_COL, "target_triage_acuity"])
    .copy()
)
data["target_triage_acuity"] = data["target_triage_acuity"].astype(int)

display(data.head(10))
print(f"Rows used: {len(data):,}")
display(data["target_triage_acuity"].value_counts().sort_index().rename("count").to_frame())

,chief_complaint_text,target_triage_acuity
0,fever. throat soreness,4
1,"injury, other and unspecified, of foo.... foot...",4
2,fever. cough,4
3,"abdominal pain, cramps, spasms. chest pain. po...",4
4,"injury, other and unspecified, of foo... ...",4
5,fever,4
6,cough. nasal congestion,3
7,cough,4
8,"injury, other and unspecified of head... ...",4
9,skin rash,3


Rows used: 58,124


,count
target_triage_acuity,
1,846
2,8597
3,29568
4,16715
5,2398


In [22]:
# Expand common short forms in chief complaint text
ABBR_MAP = {
    "sob": "shortness of breath",
    "cp": "chest pain",
    "abd": "abdominal",
    "ha": "headache",
    "n/v": "nausea and vomiting",
    "s/p": "status post",
    "w/": "with",
    "w/o": "without",
    "fx": "fracture",
    "lac": "laceration",
    "loc": "loss of consciousness",
    "mva": "motor vehicle accident",
    "uti": "urinary tract infection",
    "uri": "upper respiratory infection",
}

def expand_text(text: str) -> str:
    txt = str(text).lower().strip()
    txt = re.sub(r"\s+", " ", txt)

    # Replace slash abbreviations first.
    for k in ["n/v", "s/p", "w/o", "w/"]:
        txt = re.sub(rf"(?<!\w){re.escape(k)}(?!\w)", ABBR_MAP[k], txt)

    for abbr, full in ABBR_MAP.items():
        if "/" in abbr:
            continue
        txt = re.sub(rf"(?<![a-z0-9]){re.escape(abbr)}(?![a-z0-9])", full, txt)

    txt = re.sub(r"[^a-z0-9\s\-\.,]", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

data["text_clean"] = data["chief_complaint_text"].map(expand_text)
display(data[["chief_complaint_text", "text_clean", "target_triage_acuity"]].head(10))

,chief_complaint_text,text_clean,target_triage_acuity
0,fever. throat soreness,fever. throat soreness,4
1,"injury, other and unspecified, of foo.... foot...","injury, other and unspecified, of foo.... foot...",4
2,fever. cough,fever. cough,4
3,"abdominal pain, cramps, spasms. chest pain. po...","abdominal pain, cramps, spasms. chest pain. po...",4
4,"injury, other and unspecified, of foo... ...","injury, other and unspecified, of foo...",4
5,fever,fever,4
6,cough. nasal congestion,cough. nasal congestion,3
7,cough,cough,4
8,"injury, other and unspecified of head... ...","injury, other and unspecified of head...",4
9,skin rash,skin rash,3


In [29]:
# DistilBERT (uncased) + CORAL ordinal head with CV and QWK reporting
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "nlpie/distil-clinicalbert"
NUM_CLASSES = 5
MAX_LEN = 96
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LR = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 2
CV_FOLDS = 5
USE_FP16 = DEVICE.type == "cuda"

print(f"Device: {DEVICE}")

class OrdinalTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=96):
        self.texts = texts.astype(str).tolist()
        self.labels = labels.astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["label"] = torch.tensor(self.labels[idx] - 1, dtype=torch.long)
        return item

class OrdinalBert(nn.Module):
    def __init__(self, model_name: str, num_classes: int = 5, dropout: float = 0.2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.ordinal_head = nn.Linear(hidden, num_classes - 1)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_state = out.last_hidden_state[:, 0]
        return self.ordinal_head(self.dropout(cls_state))

def make_coral_targets(y_idx: torch.Tensor, num_classes: int = 5) -> torch.Tensor:
    thresholds = torch.arange(num_classes - 1, device=y_idx.device).unsqueeze(0)
    return (y_idx.unsqueeze(1) > thresholds).float()

def logits_to_class_idx(logits: torch.Tensor) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    return (probs > 0.5).sum(dim=1).long()

def run_epoch(model, loader, optimizer=None, scheduler=None, scaler=None, desc="epoch"):
    is_train = optimizer is not None
    model.train(is_train)
    criterion = nn.BCEWithLogitsLoss()

    all_true, all_pred = [], []
    total_loss = 0.0

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, desc=desc, leave=False)
    for batch in progress:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        y_idx = batch["label"].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=USE_FP16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            targets = make_coral_targets(y_idx, num_classes=NUM_CLASSES)
            loss = criterion(logits, targets)

        if is_train:
            if scaler is not None and USE_FP16:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()

        pred_idx = logits_to_class_idx(logits).detach().cpu().numpy()
        true_idx = y_idx.detach().cpu().numpy()
        all_pred.extend((pred_idx + 1).tolist())
        all_true.extend((true_idx + 1).tolist())
        total_loss += loss.item() * input_ids.size(0)

        avg_loss_so_far = total_loss / max(1, len(all_true))
        progress.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    epoch_loss = total_loss / len(loader.dataset)
    qwk = cohen_kappa_score(all_true, all_pred, weights="quadratic")
    return epoch_loss, qwk, all_true, all_pred

Device: cuda


In [ ]:
X = data["text_clean"].fillna("")
y = data["target_triage_acuity"].astype(int)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

fold_qwks = []
oof_pred = np.zeros(len(y), dtype=int)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\nFold {fold}/{CV_FOLDS}")
    X_train, X_val = X.iloc[tr_idx], X.iloc[va_idx]
    y_train, y_val = y.iloc[tr_idx], y.iloc[va_idx]

    train_ds = OrdinalTextDataset(X_train, y_train, tokenizer, max_len=MAX_LEN)
    val_ds = OrdinalTextDataset(X_val, y_val, tokenizer, max_len=MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

    model = OrdinalBert(MODEL_NAME, num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(train_loader) * EPOCHS)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

    best_qwk = -1.0
    best_pred = None

    for epoch in tqdm(range(1, EPOCHS + 1), desc=f"Fold {fold} epochs", leave=False):
        tr_loss, tr_qwk, _, _ = run_epoch(
            model,
            train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            desc=f"Fold {fold} train e{epoch}",
        )
        va_loss, va_qwk, _, va_pred = run_epoch(
            model,
            val_loader,
            desc=f"Fold {fold} valid e{epoch}",
        )
        print(
            f"Epoch {epoch}/{EPOCHS} | "
            f"train_loss={tr_loss:.4f} train_qwk={tr_qwk:.4f} | "
            f"val_loss={va_loss:.4f} val_qwk={va_qwk:.4f}"
        )

        if va_qwk > best_qwk:
            best_qwk = va_qwk
            best_pred = np.array(va_pred, dtype=int)

    fold_qwks.append(best_qwk)
    oof_pred[va_idx] = best_pred
    print(f"Best fold QWK: {best_qwk:.4f}")

overall_qwk = cohen_kappa_score(y, oof_pred, weights="quadratic")
print("\nCV summary")
print(f"Mean fold QWK: {np.mean(fold_qwks):.4f}")
print(f"Std fold QWK: {np.std(fold_qwks):.4f}")
print(f"OOF QWK: {overall_qwk:.4f}")

print("\nClassification report (OOF):")
print(classification_report(y, oof_pred, digits=4))

labels = [1, 2, 3, 4, 5]
cm = confusion_matrix(y, oof_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{i}" for i in labels], columns=[f"pred_{i}" for i in labels])
display(cm_df)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: nlpie/distil-clinicalbert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstr

Fold 1 epochs:   0%|          | 0/2 [00:00<?, ?it/s]

Fold 1 train e1:   0%|          | 0/5813 [00:00<?, ?it/s]

/tmp/ipykernel_31905/870709014.py:84: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):


model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]